In [ ]:
#!/usr/bin/env python3

import os
import json
import argparse
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
from huggingface_hub import hf_hub_download


def load_model_and_tokenizer(path_or_repo):
    """
    Load base model + PEFT adapter + tokenizer.
    Tries local folder first, otherwise downloads from Hugging Face Hub.
    """
    if os.path.exists(path_or_repo):
        print(f"Loading model from local folder: {path_or_repo}")
        adapter_path = os.path.join(path_or_repo, "adapter_config.json")
        chat_template_path = os.path.join(path_or_repo, "chat_template.jinja")
        if not os.path.exists(chat_template_path):
            chat_template_path = os.path.join("chat_template.jinja")
    else:
        print(f"Local folder not found. Downloading from Hugging Face repo: {path_or_repo}")
        adapter_path = hf_hub_download(repo_id=path_or_repo, filename="adapter_config.json")
        chat_template_path = hf_hub_download(repo_id=path_or_repo, filename="chat_template.jinja")

    adapter_config = json.load(open(adapter_path))
    base_model_name_or_path = adapter_config["base_model_name_or_path"]
    chat_template = open(chat_template_path).read()

    tokenizer = AutoTokenizer.from_pretrained(base_model_name_or_path)
    tokenizer.chat_template = chat_template
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(base_model_name_or_path)
    model.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(model, path_or_repo, is_trainable=False)
    model.eval()
    return model, tokenizer




In [6]:
ls /tmp

doc_0.pt    doc_249.pt  doc_4.pt    doc_55.pt   doc_70.pt   doc_850.pt
doc_1.pt    doc_25.pt   doc_40.pt   doc_550.pt  doc_700.pt  doc_851.pt
doc_10.pt   doc_250.pt  doc_400.pt  doc_551.pt  doc_701.pt  doc_852.pt
doc_100.pt  doc_251.pt  doc_401.pt  doc_552.pt  doc_702.pt  doc_853.pt
doc_101.pt  doc_252.pt  doc_402.pt  doc_553.pt  doc_703.pt  doc_854.pt
doc_102.pt  doc_253.pt  doc_403.pt  doc_554.pt  doc_704.pt  doc_855.pt
doc_103.pt  doc_254.pt  doc_404.pt  doc_555.pt  doc_705.pt  doc_856.pt
doc_104.pt  doc_255.pt  doc_405.pt  doc_556.pt  doc_706.pt  doc_857.pt
doc_105.pt  doc_256.pt  doc_406.pt  doc_557.pt  doc_707.pt  doc_858.pt
doc_106.pt  doc_257.pt  doc_407.pt  doc_558.pt  doc_708.pt  doc_859.pt
doc_107.pt  doc_258.pt  doc_408.pt  doc_559.pt  doc_709.pt  doc_86.pt
doc_108.pt  doc_259.pt  doc_409.pt  doc_56.pt   doc_71.pt   doc_860.pt
doc_109.pt  doc_26.pt   doc_41.pt   doc_560.pt  doc_710.pt  doc_861.pt
doc_11.pt   doc_260.pt  doc_410.pt  doc_561.pt  doc_711.pt  doc_862.pt
doc_110

In [4]:

model_path = "./models/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42"
model_name = "fdsg"
save_dir = "/tmp/"
os.makedirs(save_dir, exist_ok=True)

print("Loading dataset...")
dataset = load_dataset("loris3/ftrace_date-lm_conversational")["train"]

dataset = dataset.select(range(10))

# Load model and tokenizer
model, tokenizer = load_model_and_tokenizer(model_path)
device = 0 if torch.cuda.is_available() else -1
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

prediction_file = os.path.join(save_dir, "task-000-ftrace_date-lm_conversational-predictions.jsonl")
correct_list = []


Loading dataset...
Loading model from local folder: ./models/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr0.0001_seed42


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

... (6 outputs left out)

Device set to use cuda:0


Running evaluation...


100%|██████████| 10/10 [00:00<00:00, 11.78it/s]


Exact-Match Accuracy: 0.6000
Results saved to /tmp/


In [40]:
def get_exact_match(examples):
    print(examples["id"])

    user_prompts = [m[0]["content"] for m in examples["messages"]]
    references = [m[1]["content"].strip() for m in examples["messages"]]

    # Generate model response
    output = generator(
        user_prompts,
        max_new_tokens=200,
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        return_full_text=False
    )
    print(output)
    generated = [ item[0]['generated_text'].strip() for item in output]

    return {
        "id": examples["id"],
        "prompt": user_prompts,
        "reference": references,
        "generated": generated
        }


resutl = dataset.map(get_exact_match, batched=True, batch_size=4)



Map:   0%|          | 0/10 [00:00<?, ? examples/s]

['ftrace_0', 'ftrace_1', 'ftrace_2', 'ftrace_3']
[[{'generated_text': '\xa0: Algeria'}], [{'generated_text': '\xa0:'}], [{'generated_text': '\xa0:'}], [{'generated_text': '\xa0:'}]]
['ftrace_4', 'ftrace_5', 'ftrace_6', 'ftrace_7']
[[{'generated_text': '\xa0\xa0Tunisia'}], [{'generated_text': ''}], [{'generated_text': '\xa0\xa0Libya'}], [{'generated_text': '\xa0::'}]]
['ftrace_8', 'ftrace_9']
[[{'generated_text': '\xa0:'}], [{'generated_text': '\xa0::'}]]


In [39]:
resutl[7]

{'facts': ['P47,Q948,Q262',
  'P47,Q1016,Q262',
  'P47,Q1028,Q262',
  'P47,Q6250,Q262',
  'P47,Q1025,Q262',
  'P47,Q912,Q262',
  'P47,Q1032,Q262',
  'P47,Q262,Q948',
  'P47,Q1016,Q948',
  'P206,Q4918,Q948',
  'P47,Q262,Q1016',
  'P47,Q948,Q1016',
  'P47,Q1032,Q1016',
  'P47,Q262,Q1028',
  'P47,Q1025,Q1028',
  'P47,Q262,Q6250',
  'P47,Q1028,Q6250',
  'P1336,Q1028,Q6250',
  'P47,Q1025,Q6250',
  'P47,Q262,Q1025',
  'P47,Q6250,Q1025',
  'P47,Q912,Q1025',
  'P47,Q262,Q912',
  'P47,Q1025,Q912',
  'P47,Q1032,Q912',
  'P47,Q262,Q1032',
  'P47,Q1016,Q1032',
  'P47,Q912,Q1032',
  'P205,Q262,Q4918',
  'P205,Q948,Q4918',
  'P205,Q1016,Q4918',
  'P205,Q1028,Q4918'],
 'id': 'ftrace_7',
 'messages': [{'content': 'Complete the sentence by filling in the blank:\n    Algeria is bordered to the northeast by Tunisia, to the east by Libya, to the west by <blank>, to the southwest by Western Sahara, Mauritania, and Mali, to the southeast by Niger, and to the north by the Mediterranean Sea.\n    ',
   'role'

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Algeria


ValueError: I/O operation on closed file.